# Session 22 — Toxicity Gate B measurement

Per the Session 21 viability assessment (`memory_bank/concepts/toxicity/VIABILITY.md`), this notebook measures Gate B: **how many of Syn3A's 155 SBML gene-associated enzymes have ChEMBL inhibitor data with documented IC50 / Ki / Kd?**

Output: `memory_bank/data/toxicity/syn3a_enzyme_inhibitor_coverage.csv` (one row per SBML gene; schema in `memory_bank/data/toxicity/README.md`).

**Decision rule** (locked at end of Session 21):

| outcome | action |
|---|---|
| ≥ 30 enzymes with `n_inhibitors_strong ≥ 5` | proceed to Session 23 (validation set) |
| 10-30 such enzymes | proceed but scope down to a narrow validation set |
| < 10 such enzymes | **halt** — document negative finding |

**Pipeline:** SBML genes → Syn3A protein sequences → UniProt mapping (orthology to M. genitalium G37 / M. pneumoniae M129) → ChEMBL target + activity counts → coverage CSV.

Wall estimate on Colab (no GPU needed): ~10-20 minutes, dominated by ~310 UniProt + ChEMBL HTTP calls at polite rate-limited speed.

**Per the spec's hard non-negotiables:** no compound-target data is fabricated; every UniProt accession + ChEMBL target ID + inhibitor count comes from the public APIs and is auditable from the saved checkpoint files.

In [ ]:
# Cell 1 — install minimal deps. No torch / transformers — pure data work.
!pip install -q biopython>=1.83 pandas>=2 requests>=2.31 pyarrow>=14

In [ ]:
# Cell 2 — clone (or refresh) the project repo to branch tip.
import os, subprocess

BRANCH = "claude/syn3a-whole-cell-simulator-REjHC"
REPO_URL = "https://github.com/Nikku03/cell.git"
REPO_DIR = "/content/cell"

def _run(cmd, cwd=None):
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if r.stdout.strip(): print(r.stdout.rstrip())
    if r.stderr.strip(): print(r.stderr.rstrip())
    if r.returncode != 0:
        raise RuntimeError(f"{cmd!r} exit {r.returncode}")
    return r

if not os.path.isdir(REPO_DIR):
    _run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR])
else:
    _run(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR)
    _run(["git", "checkout", BRANCH], cwd=REPO_DIR)
    _run(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=REPO_DIR)

%cd /content/cell
import sys
sys.path.insert(0, "/content/cell")
sys.path.insert(0, "/content/cell/cell_sim")
_run(["git", "log", "--oneline", "-1"], cwd=REPO_DIR)

In [ ]:
# Cell 3 — prompt for GitHub PAT (masked).
import os, getpass
if os.environ.get("GITHUB_PAT", "").strip():
    print(f"GITHUB_PAT already set ({len(os.environ['GITHUB_PAT'])} chars).")
else:
    pat = getpass.getpass(
        "Paste your GitHub fine-grained PAT (input hidden): "
    ).strip()
    if not pat:
        raise ValueError("empty PAT — push later will fail")
    os.environ["GITHUB_PAT"] = pat
    print(f"GITHUB_PAT set ({len(pat)} chars).")

In [ ]:
# Cell 4 — load Syn3A SBML genes + GenBank protein sequences.
#
# We need three things per SBML gene:
#   1. Locus tag (JCVISYN3A_NNNN — derived from G_MMSYN1_NNNN)
#   2. Reaction(s) it catalyzes (for downstream interpretability)
#   3. Protein sequence + gene_name + product description from GenBank
import urllib.request
from collections import defaultdict
from pathlib import Path

import pandas as pd
from Bio import SeqIO

from cell_sim.layer3_reactions.sbml_parser import (
    parse_sbml, sbml_gene_to_locus,
)

STAGED = Path("cell_sim/data/Minimal_Cell_ComplexFormation/input_data")
STAGED.mkdir(parents=True, exist_ok=True)
GB = STAGED / "syn3A.gb"
SBML_PATH = STAGED / "Syn3A_updated.xml"
for fname in ("syn3A.gb", "Syn3A_updated.xml"):
    p = STAGED / fname
    if not p.exists():
        url = (
            "https://raw.githubusercontent.com/Luthey-Schulten-Lab/"
            "Minimal_Cell_ComplexFormation/master/input_data/" + fname
        )
        print(f"staging {fname} from upstream...")
        urllib.request.urlretrieve(url, p)

# 1. SBML inventory.
sbml = parse_sbml(SBML_PATH)
gene_to_rxns: dict[str, list[str]] = defaultdict(list)
for rxn in sbml.reactions.values():
    for g in rxn.gene_associations:
        gene_to_rxns[g].append(rxn.short_name)
print(f"SBML reactions: {len(sbml.reactions)}")
print(f"SBML genes with associations: {len(gene_to_rxns)}")

# 2. GenBank: locus_tag -> (gene_name, product, sequence).
rec = next(SeqIO.parse(GB, "genbank"))
gb_by_locus: dict[str, dict] = {}
for f in rec.features:
    if f.type != "CDS":
        continue
    lt = (f.qualifiers.get("locus_tag") or [""])[0]
    if not lt:
        continue
    gb_by_locus[lt] = {
        "gene_name": (f.qualifiers.get("gene") or [""])[0],
        "product": (f.qualifiers.get("product") or [""])[0],
        "sequence": (
            (f.qualifiers.get("translation") or [""])[0]
            .upper().rstrip("*")
        ),
    }
print(f"GenBank CDS with translations: {len(gb_by_locus)}")

# 3. Join.
rows = []
for sbml_gene_id in sorted(gene_to_rxns):
    locus = sbml_gene_to_locus(sbml_gene_id)
    gb = gb_by_locus.get(locus or "", {})
    rows.append({
        "sbml_gene_id": sbml_gene_id,
        "sbml_locus_tag": locus or "",
        "reaction_short_names": ";".join(sorted(set(gene_to_rxns[sbml_gene_id]))),
        "gene_name": gb.get("gene_name", ""),
        "product_description": gb.get("product", ""),
        "sequence": gb.get("sequence", ""),
        "sequence_length": len(gb.get("sequence", "")),
    })
df_genes = pd.DataFrame(rows)
n_with_seq = int((df_genes["sequence_length"] > 0).sum())
print(f"\nSBML genes: {len(df_genes)}  (with GenBank seq: {n_with_seq})")
print(df_genes.head())

In [ ]:
# Cell 5 — UniProt orthology mapping.
#
# Strategy per gene, in order. Stop at the first hit:
#   1. Search by gene_name in M. genitalium G37 (taxonomy_id:243273)
#   2. Search by gene_name in M. pneumoniae M129 (taxonomy_id:272634)
#   3. Search by product description (full-text) in M. genitalium
# Falls back gracefully — empty uniprot_accession means "unmapped".
#
# Checkpoints to /content/uniprot_map.json so a kernel restart resumes.
import json
import time
import urllib.parse
import urllib.request

from pathlib import Path

UNIPROT_BASE = "https://rest.uniprot.org/uniprotkb/search"
TAXIDS = [243273, 272634]   # M. genitalium G37, M. pneumoniae M129
TAX_NAMES = {243273: "mgenitalium_ortholog", 272634: "mpneumoniae_ortholog"}
FIELDS = "accession,gene_names,organism_id,length,protein_name"

CKPT_PATH = Path("/content/uniprot_map.json")
if CKPT_PATH.exists():
    cache = json.loads(CKPT_PATH.read_text())
else:
    cache = {}
print(f"[resume] uniprot cache: {len(cache)} entries")

def _http_get(url, timeout=30):
    req = urllib.request.Request(
        url, headers={"User-Agent": "Mozilla/5.0"},
    )
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return resp.read().decode("utf-8", errors="replace")

def _query_uniprot(query: str) -> list[dict]:
    params = urllib.parse.urlencode({
        "query": query,
        "format": "tsv",
        "fields": FIELDS,
        "size": "5",
    })
    url = f"{UNIPROT_BASE}?{params}"
    body = _http_get(url)
    lines = body.strip().splitlines()
    if len(lines) <= 1:
        return []
    header = lines[0].split("\t")
    out = []
    for L in lines[1:]:
        cells = L.split("\t")
        if len(cells) >= len(header):
            out.append(dict(zip(header, cells)))
    return out

def _map_one(row: dict) -> dict:
    gene_name = (row.get("gene_name") or "").strip()
    product = (row.get("product_description") or "").strip()
    if not gene_name and not product:
        return {"uniprot_accession": "", "mapping_method": "unmapped"}

    for taxid in TAXIDS:
        if gene_name:
            try:
                hits = _query_uniprot(
                    f"gene:{gene_name} AND taxonomy_id:{taxid}"
                )
            except Exception as exc:  # noqa: BLE001
                print(f"    [warn] uniprot gene query for {gene_name}/{taxid}: "
                      f"{type(exc).__name__}: {exc}")
                hits = []
            if hits:
                acc = hits[0].get("Entry") or ""
                if acc:
                    return {
                        "uniprot_accession": acc,
                        "mapping_method": TAX_NAMES[taxid] + "_by_gene",
                    }
        time.sleep(0.3)  # polite rate limit

    if product:
        for taxid in TAXIDS:
            words = [w for w in product.split() if len(w) > 3][:3]
            q = " ".join(words) if words else product[:40]
            try:
                hits = _query_uniprot(
                    f'"{q}" AND taxonomy_id:{taxid}'
                )
            except Exception as exc:  # noqa: BLE001
                hits = []
            if hits:
                acc = hits[0].get("Entry") or ""
                if acc:
                    return {
                        "uniprot_accession": acc,
                        "mapping_method": TAX_NAMES[taxid] + "_by_product",
                    }
            time.sleep(0.3)

    return {"uniprot_accession": "", "mapping_method": "unmapped"}

# Iterate; checkpoint every 25 genes.
for i, row in df_genes.iterrows():
    sbml_id = row["sbml_gene_id"]
    if sbml_id in cache:
        continue
    res = _map_one(row.to_dict())
    cache[sbml_id] = res
    n = len(cache)
    if n % 25 == 0 or n == len(df_genes):
        CKPT_PATH.write_text(json.dumps(cache, indent=2))
        n_mapped = sum(1 for v in cache.values() if v["uniprot_accession"])
        print(f"  [{n:3d}/{len(df_genes)}] {sbml_id} {row['gene_name'] or '(no gene)':12s}"
              f" -> {res['uniprot_accession'] or '-':12s} ({res['mapping_method']})"
              f"  cumulative mapped={n_mapped}")

n_mapped = sum(1 for v in cache.values() if v["uniprot_accession"])
print(f"\n[uniprot] mapped {n_mapped}/{len(df_genes)}  "
      f"({n_mapped/len(df_genes):.1%})")

df_genes["uniprot_accession"] = df_genes["sbml_gene_id"].map(
    lambda g: cache.get(g, {}).get("uniprot_accession", "")
)
df_genes["mapping_method"] = df_genes["sbml_gene_id"].map(
    lambda g: cache.get(g, {}).get("mapping_method", "unmapped")
)
df_genes["mapping_method"].value_counts().to_string()

In [ ]:
# Cell 6 — ChEMBL: target lookup + inhibitor counts.
#
# For each mapped UniProt accession:
#   a. /target.json?target_components__accession=<UNIPROT> -> target_chembl_id
#   b. /activity.json?target_chembl_id=<TID>
#         &standard_type__in=IC50,Ki,Kd,EC50
#         &pchembl_value__isnull=false
#         &limit=1000
#      paged until the response is exhausted; counts are split into
#      'strong' (pchembl >= 5, i.e. 10 uM or better) and 'weak'.
#
# Checkpoint to /content/chembl_map.json.
import json, time, urllib.parse, urllib.request
from pathlib import Path

CHEMBL_BASE = "https://www.ebi.ac.uk/chembl/api/data"
STRONG_PCHEMBL = 5.0
MAX_ACTIVITIES_PER_TARGET = 5000

CHEMBL_CKPT = Path("/content/chembl_map.json")
if CHEMBL_CKPT.exists():
    chembl_cache = json.loads(CHEMBL_CKPT.read_text())
else:
    chembl_cache = {}
print(f"[resume] chembl cache: {len(chembl_cache)} entries")

def _http_get_json(url, timeout=30):
    req = urllib.request.Request(
        url, headers={
            "User-Agent": "Mozilla/5.0",
            "Accept": "application/json",
        },
    )
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return json.loads(resp.read().decode("utf-8"))

def _chembl_target_for_uniprot(uniprot_acc: str) -> str:
    if not uniprot_acc:
        return ""
    url = (
        f"{CHEMBL_BASE}/target.json?"
        + urllib.parse.urlencode({
            "target_components__accession": uniprot_acc,
            "limit": "5",
        })
    )
    try:
        body = _http_get_json(url)
    except Exception as exc:  # noqa: BLE001
        print(f"    [warn] chembl target {uniprot_acc}: "
              f"{type(exc).__name__}: {exc}")
        return ""
    targets = body.get("targets", [])
    if not targets:
        return ""
    return targets[0].get("target_chembl_id", "")

def _chembl_activity_counts(target_id: str) -> dict:
    if not target_id:
        return {
            "n_inhibitors_strong": 0,
            "n_inhibitors_weak": 0,
            "mean_pchembl_strong": None,
        }
    n_strong = 0
    n_weak = 0
    sum_pchembl = 0.0
    n_pchembl = 0
    seen = 0
    offset = 0
    page_size = 1000
    while seen < MAX_ACTIVITIES_PER_TARGET:
        url = (
            f"{CHEMBL_BASE}/activity.json?"
            + urllib.parse.urlencode({
                "target_chembl_id": target_id,
                "standard_type__in": "IC50,Ki,Kd,EC50",
                "pchembl_value__isnull": "false",
                "limit": str(page_size),
                "offset": str(offset),
            })
        )
        try:
            body = _http_get_json(url, timeout=60)
        except Exception as exc:  # noqa: BLE001
            print(f"    [warn] chembl activity {target_id} offset={offset}: "
                  f"{type(exc).__name__}: {exc}")
            break
        acts = body.get("activities", [])
        if not acts:
            break
        for a in acts:
            try:
                pv = float(a.get("pchembl_value") or "nan")
            except (TypeError, ValueError):
                continue
            if pv != pv:   # NaN
                continue
            if pv >= STRONG_PCHEMBL:
                n_strong += 1
                sum_pchembl += pv
                n_pchembl += 1
            else:
                n_weak += 1
        seen += len(acts)
        offset += page_size
        meta = body.get("page_meta", {})
        if not meta.get("next"):
            break
        time.sleep(0.3)
    return {
        "n_inhibitors_strong": n_strong,
        "n_inhibitors_weak": n_weak,
        "mean_pchembl_strong": (
            sum_pchembl / n_pchembl if n_pchembl else None
        ),
    }

for i, row in df_genes.iterrows():
    sbml_id = row["sbml_gene_id"]
    if sbml_id in chembl_cache:
        continue
    acc = row.get("uniprot_accession") or ""
    if not acc:
        chembl_cache[sbml_id] = {
            "chembl_target_id": "",
            "n_inhibitors_strong": 0,
            "n_inhibitors_weak": 0,
            "mean_pchembl_strong": None,
        }
        continue
    tid = _chembl_target_for_uniprot(acc)
    time.sleep(0.3)
    counts = _chembl_activity_counts(tid)
    chembl_cache[sbml_id] = {"chembl_target_id": tid, **counts}
    n = len(chembl_cache)
    if n % 10 == 0 or n == len(df_genes):
        CHEMBL_CKPT.write_text(json.dumps(chembl_cache, indent=2))
    print(f"  [{n:3d}/{len(df_genes)}] {sbml_id} {row['gene_name'] or '?':10s}"
          f" uniprot={acc:8s} chembl={tid or '-':12s}"
          f" strong={counts['n_inhibitors_strong']:<5d}"
          f" weak={counts['n_inhibitors_weak']:<5d}")

CHEMBL_CKPT.write_text(json.dumps(chembl_cache, indent=2))
print("\n[chembl] queries complete.")

In [ ]:
# Cell 7 — assemble coverage CSV + headline number.
from pathlib import Path
import pandas as pd

for col in (
    "chembl_target_id", "n_inhibitors_strong",
    "n_inhibitors_weak", "mean_pchembl_strong",
):
    df_genes[col] = df_genes["sbml_gene_id"].map(
        lambda g: chembl_cache.get(g, {}).get(col)
    )

out_cols = [
    "sbml_gene_id", "sbml_locus_tag", "reaction_short_names",
    "gene_name", "product_description", "sequence_length",
    "uniprot_accession", "mapping_method",
    "chembl_target_id",
    "n_inhibitors_strong", "n_inhibitors_weak", "mean_pchembl_strong",
]
out = df_genes[out_cols].copy()
out["n_inhibitors_strong"] = out["n_inhibitors_strong"].fillna(0).astype(int)
out["n_inhibitors_weak"] = out["n_inhibitors_weak"].fillna(0).astype(int)

OUT_DIR = Path("memory_bank/data/toxicity")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = OUT_DIR / "syn3a_enzyme_inhibitor_coverage.csv"
out.to_csv(OUT_CSV, index=False)
print(f"wrote {OUT_CSV}  rows={len(out)}")

# Headline.
n_total = len(out)
n_uniprot_mapped = int((out["uniprot_accession"] != "").sum())
n_chembl_target = int((out["chembl_target_id"] != "").fillna(False).sum())
n_strong_5 = int((out["n_inhibitors_strong"] >= 5).sum())
n_strong_50 = int((out["n_inhibitors_strong"] >= 50).sum())

print("\n=== Gate B headline ===")
print(f"SBML genes total                       : {n_total}")
print(f"  with UniProt accession               : {n_uniprot_mapped}")
print(f"  with ChEMBL target                   : {n_chembl_target}")
print(f"  with >=5  strong inhibitors          : {n_strong_5}")
print(f"  with >=50 strong inhibitors          : {n_strong_50}")

if n_strong_5 >= 30:
    decision = "PROCEED — full validation set (>=30 enzymes)"
elif n_strong_5 >= 10:
    decision = "PROCEED — narrow validation set (10-30 enzymes)"
else:
    decision = "HALT — < 10 enzymes; document negative finding"
print(f"\n=== decision: {decision} ===")

# Top-15 most-targeted enzymes — useful for the user to eyeball
top15 = out.sort_values("n_inhibitors_strong", ascending=False).head(15)
print("\ntop 15 enzymes by strong-inhibitor count:")
for _, r in top15.iterrows():
    print(f"  {r['sbml_locus_tag']:18s} {r['gene_name'] or '?':10s} "
          f"strong={r['n_inhibitors_strong']:<5d} "
          f"uniprot={r['uniprot_accession']:8s} "
          f"product={r['product_description'][:60]}")

In [ ]:
# Cell 8 — auto-commit + push.
import os, subprocess
from pathlib import Path

pat = os.environ.get("GITHUB_PAT", "").strip()
if not pat:
    raise SystemExit("GITHUB_PAT not set; re-run Cell 3.")

BRANCH = "claude/syn3a-whole-cell-simulator-REjHC"
MSG = (
    "Session 22: Gate B coverage CSV "
    "(syn3a_enzyme_inhibitor_coverage.csv)"
)

def run(cmd, check=True):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.stdout.strip(): print(r.stdout.rstrip())
    if r.stderr.strip(): print(r.stderr.rstrip())
    if check and r.returncode != 0:
        raise RuntimeError(f"{cmd!r} exit {r.returncode}")
    return r

run(["git", "config", "user.email", "cell-sim-bot@noreply.local"])
run(["git", "config", "user.name", "cell-sim-bot"])
run(["git", "add", "-f",
     "memory_bank/data/toxicity/syn3a_enzyme_inhibitor_coverage.csv"])

status = run(["git", "status", "--porcelain"], check=False)
if not status.stdout.strip():
    print("nothing changed")
else:
    run(["git", "commit", "-m", MSG])
    remote = f"https://{pat}@github.com/Nikku03/cell.git"
    run(["git", "push", remote, BRANCH])
    print("\npush complete.")

# Clean up checkpoints once the CSV is on origin.
for p in (Path("/content/uniprot_map.json"),
          Path("/content/chembl_map.json")):
    if p.exists():
        p.unlink()

run(["git", "log", "--oneline", "-3"])